# Encoding Strategy Comparison
1. Take a dataset with at least 3 categorical columns of varying cardinality (low, medium, high).
1. Apply Label Encoding, One-Hot Encoding, and Ordinal Encoding (where appropriate) to each.
1. Train the same Logistic Regression model on each encoded version.
1. Measure and compare a ccuracy, training time, and feature count for each strategy.
1. Write a 1-paragraph recommendation on which encoding approach to use and why? backed by your results

In [1]:
%pip install scikit-learn numpy pandas


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# 1. Create a synthetic dataset with varying cardinality
np.random.seed(42)
n_samples = 5000

# Low cardinality: 2 categories (e.g., Gender)
gender = np.random.choice(['Male', 'Female'], n_samples)

# Medium cardinality: 5 categories (e.g., Education Level)
education = np.random.choice(['High School', 'Bachelors', 'Masters', 'PhD', 'Associate'], n_samples)

# High cardinality: 50 categories (e.g., US State)
states = [f"State_{i}" for i in range(1, 51)]
state = np.random.choice(states, n_samples)

# Numerical features
income = np.random.normal(50000, 15000, n_samples)
age = np.random.randint(18, 70, n_samples)

df = pd.DataFrame({
    'gender': gender,
    'education': education,
    'state': state,
    'income': income,
    'age': age
})

# Create target variable (purchased a premium product)
# The probability depends on income, age, education, and slightly on state
education_score = {'High School': 0, 'Associate': 1, 'Bachelors': 2, 'Masters': 3, 'PhD': 4}
df['edu_score'] = df['education'].map(education_score)
state_score = {s: np.random.uniform(-0.5, 0.5) for s in states}
df['state_score'] = df['state'].map(state_score)

prob = (
    0.1 + 
    (df['income'] / 100000) * 0.3 + 
    (df['age'] / 100) * 0.2 +
    (df['edu_score'] / 4) * 0.3 +
    df['state_score'] * 0.2 +
    np.random.normal(0, 0.1, n_samples)
)

df['target'] = (prob > np.median(prob)).astype(int)
df = df.drop(columns=['edu_score', 'state_score'])

print(f"Dataset created with {n_samples} samples.")
print(df.head())

Dataset created with 5000 samples.
   gender    education     state        income  age  target
0    Male      Masters   State_2  44430.056975   57       1
1  Female          PhD  State_16  49757.542481   23       0
2    Male    Associate  State_39  45624.530814   42       1
3    Male      Masters  State_13  52399.993187   20       0
4    Male  High School  State_37  68327.270292   35       0


In [3]:
# Setup evaluation tracking
results = []

# Separate features and target
X = df.drop('target', axis=1)
y = df['target']

# Continuous features to scale
continuous_cols = ['income', 'age']
categorical_cols = ['gender', 'education', 'state']

def evaluate_model(X_encoded, y, encoding_name):
    # Scale continuous features
    scaler = StandardScaler()
    X_encoded_scaled = X_encoded.copy()
    X_encoded_scaled[continuous_cols] = scaler.fit_transform(X_encoded_scaled[continuous_cols])
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(X_encoded_scaled, y, test_size=0.2, random_state=42)
    
    # Train and measure time
    model = LogisticRegression(max_iter=1000, random_state=42)
    
    start_time = time.time()
    model.fit(X_train, y_train)
    end_time = time.time()
    
    # Predict and evaluate
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    training_time = end_time - start_time
    feature_count = X_train.shape[1]
    
    results.append({
        'Encoding Strategy': encoding_name,
        'Accuracy': accuracy,
        'Training Time (s)': training_time,
        'Feature Count': feature_count
    })
    
    print(f"--- {encoding_name} ---")
    print(f"Accuracy: {accuracy:.4f} | Features: {feature_count} | Time: {training_time:.4f}s\n")

# ==============================================================================
# STRATEGY 1: Label Encoding (Assigns an integer 0 to N-1 to each category)
# Note: Label encoding is generally meant for targets, but often misused on features.
# ==============================================================================
X_label = X.copy()
for col in categorical_cols:
    le = LabelEncoder()
    X_label[col] = le.fit_transform(X_label[col])

evaluate_model(X_label, y, "Label Encoding")

# ==============================================================================
# STRATEGY 2: Ordinal Encoding (Assigns an ordered integer to categories)
# For state and gender, the order is arbitrary.
# ==============================================================================
X_ordinal = X.copy()
oe = OrdinalEncoder()
X_ordinal[categorical_cols] = oe.fit_transform(X_ordinal[categorical_cols])

evaluate_model(X_ordinal, y, "Ordinal Encoding")

# ==============================================================================
# STRATEGY 3: One-Hot Encoding (Creates a binary column for each category)
# ==============================================================================
X_ohe = X.copy()
X_ohe = pd.get_dummies(X_ohe, columns=categorical_cols, drop_first=True)

evaluate_model(X_ohe, y, "One-Hot Encoding")

# Summary Table
results_df = pd.DataFrame(results)
print("="*60)
print("FINAL COMPARISON")
print("="*60)
print(results_df.to_string(index=False))

--- Label Encoding ---
Accuracy: 0.7110 | Features: 5 | Time: 0.0098s

--- Ordinal Encoding ---
Accuracy: 0.7110 | Features: 5 | Time: 0.0060s

--- One-Hot Encoding ---
Accuracy: 0.7970 | Features: 56 | Time: 0.0120s

FINAL COMPARISON
Encoding Strategy  Accuracy  Training Time (s)  Feature Count
   Label Encoding     0.711           0.009843              5
 Ordinal Encoding     0.711           0.006002              5
 One-Hot Encoding     0.797           0.011979             56


### Recommendation
Based on the results, **One-Hot Encoding** is the most robust and recommended strategy for linear models like Logistic Regression. Label Encoding and Ordinal Encoding map categories to arbitrary integers (e.g., State_1 = 1, State_50 = 50), which incorrectly implies an ordered, numeric relationship to the model where none exists (State_50 is not 50 times greater than State_1). This typically results in lower accuracy. While One-Hot Encoding significantly increases the feature count (due to high cardinality features like 'state') and slightly increases training time, it allows the Logistic Regression model to assign independent weights to each category, consistently yielding the highest accuracy. If feature space or memory becomes an absolute bottleneck due to extremely high cardinality, one could explore Target Encoding or grouping rare categories as alternatives, but for standard scenarios, One-Hot Encoding should be the default for categorical features.